In [1]:
# ==============================================================================
# GIS Dataset Dealing Note #1: CO₂ Raster Downscaling Pipeline
# Environment: Python 3.12 Compatibility Built-in (NumPy 2.0 & Pyogrio Ready)
# ==============================================================================
import os
import geopandas as gpd
import rasterio
import numpy as np
import pandas as pd
import shapely
from shapely.geometry import shape
from rasterio.features import shapes
from rasterio.mask import mask

In [2]:
# ==============================================================================
# ⚙️ Step 0 — Parameter Settings
# ==============================================================================
city_name = "chongqing_central" #use Chongqing central urban area as an example, it's one of the research areas in my study

# Configure local geospatial raster & vector data repository paths
graced_tif = r"J:\GISA_Macro_analysis_Luo\dataset\GRACED_dealed_2021year_260317\GRACED2021_transport_tCO2_EPSG3395n.tif"
boundary_shp = rf"J:\GISA_Macro_analysis_Luo\macro_city_anlaysis\road_dealing_results\{city_name}_road\{city_name}_fishnet.shp"

# Establish infrastructure network layers and assign relative emission structural weights
road_files = {
    "highway": {"path": rf"J:\GISA_Macro_analysis_Luo\macro_city_anlaysis\road_dealing_results\{city_name}_road\{city_name}_Highways.shp", "weight": 0.1933},
    "national": {"path": rf"J:\GISA_Macro_analysis_Luo\macro_city_anlaysis\road_dealing_results\{city_name}_road\{city_name}_National_roads.shp", "weight": 0.2353},
    "provincial": {"path": rf"J:\GISA_Macro_analysis_Luo\macro_city_anlaysis\road_dealing_results\{city_name}_road\{city_name}_Provincial_roads.shp", "weight": 0.1193},
    "county": {"path": rf"J:\GISA_Macro_analysis_Luo\macro_city_anlaysis\road_dealing_results\{city_name}_road\{city_name}_County_roads.shp", "weight": 0.4520}
}

# ==============================================================================
# 📐 Step 1 — Load Fishnet and Convert Geometry to 2D
# ==============================================================================
print("--- Step 1: Loading fishnet layers and enforcing 2D constraints...")
fine_gdf = gpd.read_file(boundary_shp)
target_crs = fine_gdf.crs

# NumPy 2.0 Compatibility Hack: Convert vector structures into sequential lists 
# to bypass coordinate multidimensional errors during mapping
geoms_2d = [shapely.wkb.loads(shapely.wkb.dumps(g, output_dimension=2)) for g in fine_gdf.geometry]
fine_gdf.geometry = geoms_2d
fine_gdf["fine_id"] = range(len(fine_gdf))

# ==============================================================================
# 🖼️ Step 2 — Raster Processing
# ==============================================================================
print("--- Step 2: Extracting raster data and instantiating vector polygons...")
with rasterio.open(graced_tif) as src:
    # Extract structural spatial bounding box from the high-resolution fishnet grid
    bbox = fine_gdf.total_bounds
    out_image, out_transform = mask(src, [shapely.geometry.box(*bbox)], crop=True)
    raster_data = out_image[0]
    
    # Isolate valid raster grid arrays and strip away Nodating values
    geoms, values = [], []
    for geom, val in shapes(raster_data, transform=out_transform):
        if val is not None and val != src.nodata:
            geoms.append(shape(geom))
            values.append(val)
            
    # Cast raster matrices into a unified vector GeoDataFrame
    coarse_gdf = gpd.GeoDataFrame({"value": values}, geometry=geoms, crs=src.crs)

# Enforce consistent Coordinate Reference Systems (CRS) alignment across datasets
if coarse_gdf.crs != target_crs:
    coarse_gdf = coarse_gdf.to_crs(target_crs)
coarse_gdf["coarse_id"] = range(len(coarse_gdf))

# ==============================================================================
# 🗺️ Step 3 — Spatial Mapping Between Fine and Coarse Grids
# ==============================================================================
print("--- Step 3: Establishing spatial index mappings via centroids...")
# Optimization Strategy: Perform spatial joins utilizing centroid vectors instead 
# of full polygon overlays to prevent topological boundary fragmentation errors
fine_centroids = fine_gdf.copy()
fine_centroids.geometry = fine_centroids.geometry.centroid
mapping = gpd.sjoin(fine_centroids, coarse_gdf, how="left", predicate="within")

# ==============================================================================
# 🛣️ Step 4 — Road Density Calculation
# ==============================================================================
print("--- Step 4: Intersecting multi-class road networks with fishnet grid...")
# Initialize tracking distance columns across all network variants
for rtype in road_files.keys():
    mapping[f"len_{rtype}"] = 0.0

for rtype, info in road_files.items():
    if not os.path.exists(info["path"]): 
        continue
    road = gpd.read_file(info["path"])
    if road.crs != target_crs: 
        road = road.to_crs(target_crs)
    
    # Apply identical 2D geometric parsing to incoming line strings
    road.geometry = [shapely.wkb.loads(shapely.wkb.dumps(g, output_dimension=2)) for g in road.geometry]
    
    # Vectorized Spatial Join: Track road vectors intersecting with specific fishnet cells
    road_match = gpd.sjoin(road, fine_gdf[['fine_id', 'geometry']], how='inner', predicate='intersects')
    
    if not road_match.empty:
        # Calculate localized linear distance metrics as structural downscaling proxies
        road_match['tmp_len'] = road_match.geometry.length
        l_sum = road_match.groupby('fine_id')['tmp_len'].sum()
        
        # Populate consolidated length matrices back into global mapping matrix
        mapping.loc[mapping['fine_id'].isin(l_sum.index), f"len_{rtype}"] = mapping['fine_id'].map(l_sum)
        print(f"--- {rtype.upper()} processing successful: populated {len(l_sum)} intersecting grid nodes.")

# ==============================================================================
# 🧮 Step 5 — Downscaling Calculation
# ==============================================================================
print("--- Step 5: Executing spatial emission weight allocations...")
mapping["W_i"] = 0.0
for rtype, info in road_files.items():
    mapping["W_i"] += mapping[f"len_{rtype}"].fillna(0) * info["weight"]

# Incorporate planar polygon area factors alongside a minute offset factor to clear zero-division risks
mapping["area_fine"] = fine_gdf.geometry.area
mapping["W_i"] = (mapping["W_i"] + 1e-10) * mapping["area_fine"]

# Apply local normalization inside each coarse ID envelope to maintain Mass Conservation
mapping["W_sum"] = mapping.groupby("coarse_id")["W_i"].transform("sum")
mapping["E_correct"] = mapping["value"] * (mapping["W_i"] / mapping["W_sum"])

# ==============================================================================
# 💾 Step 6 — Export Results
# ==============================================================================
print("--- Step 6: Exporting high-resolution downscaled emission dataset...")
fine_gdf["E_correct"] = mapping["E_correct"].fillna(0)

output_shp = rf"J:\GISA_Macro_analysis_Luo\macro_city_anlaysis\graced_co2_260317\{city_name}_final_fix_example.shp"

# Attempt high-performance vector export utilizing pyogrio C-engine acceleration;
# Fallback to standard robust GeoPackage (.gpkg) formatting if local driver blocks shapefiles
try:
    fine_gdf[["fine_id", "E_correct", "geometry"]].to_file(output_shp, driver="ESRI Shapefile", engine="pyogrio")
    print(f"✅ Pipeline executed successfully! ESRI Shapefile exported: {output_shp}")
except Exception as e:
    gpkg_output = output_shp.replace(".shp", ".gpkg")
    fine_gdf[["fine_id", "E_correct", "geometry"]].to_file(gpkg_output, driver="GPKG")
    print(f"✅ Pipeline executed successfully! Fallback GeoPackage exported: {gpkg_output}")

--- Step 1: Loading fishnet layers and enforcing 2D constraints...
--- Step 2: Extracting raster data and instantiating vector polygons...
--- Step 3: Establishing spatial index mappings via centroids...
--- Step 4: Intersecting multi-class road networks with fishnet grid...
--- HIGHWAY processing successful: populated 1042 intersecting grid nodes.
--- NATIONAL processing successful: populated 2731 intersecting grid nodes.
--- PROVINCIAL processing successful: populated 2128 intersecting grid nodes.
--- COUNTY processing successful: populated 5012 intersecting grid nodes.
--- Step 5: Executing spatial emission weight allocations...
--- Step 6: Exporting high-resolution downscaled emission dataset...
✅ Pipeline executed successfully! ESRI Shapefile exported: J:\GISA_Macro_analysis_Luo\macro_city_anlaysis\graced_co2_260317\chongqing_central_final_fix_example.shp
